# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
meta_json = metadata.to_json()
print(f"Dataset title: {meta_json['name']}")
print(f"Description: {meta_json['description']}")
print(f"Published: {meta_json['datePublished']}")

## 2. Data Overview
Review available Record Sets, Fields, and their `@id`s within the Croissant schema.

In [ ]:
# List all Record Sets and their IDs
print('Available Record Sets:')
for rs in metadata.record_sets:
    print(f"  RecordSet @id: {rs.id} | name: {rs.name}")

# For each record set, list the fields and columns by @id
print("\nFields and columns for each RecordSet:")
for rs in metadata.record_sets:
    print(f"\nRecordSet: {rs.name} (@id: {rs.id})")
    print("  Fields:")
    for field in rs.fields:
        print(f"    - {field.name} (@id: {field.id}) | dataType: {field.data_type if hasattr(field, 'data_type') else 'unknown'}")
        if getattr(field, 'columns', []):
            print("      Columns:")
            for col in field.columns:
                print(f"        * {col.name} (@id: {col.id})")

## 3. Data Extraction
Load data from one or more record sets into pandas DataFrames for analysis.
Reference all entities using their Croissant `@id` values.

In [ ]:
# Prepare to extract dataframes for each record set
record_sets_ids = [rs.id for rs in dataset.metadata.record_sets]
dataframes = {}

for record_set_id in record_sets_ids:
    print(f"Loading records from RecordSet: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f" -- Columns: {df.columns.tolist()}")

# For demonstration, pick the first record set for further operations
main_rs_id = record_sets_ids[0] if record_sets_ids else None
if main_rs_id:
    print(f"\nFirst few records from {main_rs_id}:")
    display(dataframes[main_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Demonstration of data filtering, normalization, and grouping using only `@id` to reference fields.

- Select a numeric field (using its `@id`)
- Filter all rows with that field above a chosen threshold
- Normalize the field
- Optionally group by a categorical field

*Note: Please replace the selected `numeric_field_id` and `group_field_id` with the actual `@id`s listed above for your dataset!*

In [ ]:
# Example: Assume the main record set has a field with @id 'accession', 'peptide_count', 'molecular_weight', etc.
# Replace these with correct values from the overview section. We'll use the first numeric column found.

# Find a numeric field to analyze
import numpy as np

numeric_field_id = None
group_field_id = None

if main_rs_id is not None and not dataframes[main_rs_id].empty:
    sample_df = dataframes[main_rs_id]
    # Try to infer numeric fields
    for c in sample_df.columns:
        # basic type detection
        try:
            vals = pd.to_numeric(sample_df[c], errors='coerce')
            if vals.notna().sum() / len(sample_df) > 0.5 and vals.nunique() > 10:
                numeric_field_id = c
                break
        except Exception:
            continue
    # Try to infer a group field (categorical with low cardinality)
    for c in sample_df.columns:
        if sample_df[c].dtype == object and sample_df[c].nunique() < min(20, sample_df.shape[0]//2):
            group_field_id = c
            break

if numeric_field_id is None:
    print('No numeric field found to analyze.')
else:
    print(f"Analyzing numeric field '@id': {numeric_field_id}")
    df = sample_df.copy()
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    threshold = df[numeric_field_id].mean() if not np.isnan(df[numeric_field_id].mean()) else 0
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    display(filtered_df.head())
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / (filtered_df[numeric_field_id].std() + 1e-9)
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        print(f"Grouped by {group_field_id}:")
        display(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset using `@id` references.

In [ ]:
# Histogram and scatterplot (if enough numeric columns)
import matplotlib.pyplot as plt
import seaborn as sns

if main_rs_id is not None and numeric_field_id is not None:
    fig, ax = plt.subplots(1,2, figsize=(12, 4))
    # Histogram
    filtered_df[numeric_field_id].plot(kind='hist', ax=ax[0], bins=20, color='skyblue')
    ax[0].set_title(f"Histogram of {numeric_field_id}")
    ax[0].set_xlabel(numeric_field_id)

    # If another numeric field exists, make a scatter plot
    numeric_cols = [col for col in filtered_df.columns if pd.api.types.is_numeric_dtype(filtered_df[col]) and col!=numeric_field_id]
    if numeric_cols:
        second_numeric = numeric_cols[0]
        sns.scatterplot(data=filtered_df, x=numeric_field_id, y=second_numeric, ax=ax[1])
        ax[1].set_title(f"{numeric_field_id} vs {second_numeric}")
    else:
        ax[1].remove()
    plt.tight_layout()
    plt.show()
else:
    print('Insufficient data for visualization.')

## 6. Conclusion
This notebook guided you through loading, exploring, and analyzing the FAIR² dataset on extracellular vesicles isolated from stimulated human mast cells using `mlcroissant`.

- Dataset metadata is accessible via the Croissant schema and all entities (record sets, fields, columns) are referenced by their `@id` values, enabling reproducible workflows.
- You inspected and visualized key numeric features, prepared data by filtering and normalization, and grouped by attributes to explore structure and content.

You can further extend this analysis by exploring additional record sets or fields as relevant to your research.